# Unit 05: Attention Scores

This unit connects dot product intuition to attention. Attention scores are dot products between query and key vectors after learned projections.

## Review Focus

Keep these ideas in view:

- `X` contains token representations.
- `W_Q` maps token representations into query vectors.
- `W_K` maps token representations into key vectors.
- `Q @ K.T` compares every query token against every key token.
- The output shape `[T, T]` is a token-by-token score grid.
- Scaling by `sqrt(d_head)` keeps dot products from growing too large as the head dimension grows.
- Softmax later turns raw scores into routing weights.

## Core Shapes

For one sequence:

```text
X:        [T, d_model]
W_Q:      [d_model, d_head]
W_K:      [d_model, d_head]
Q = X @ W_Q              -> [T, d_head]
K = X @ W_K              -> [T, d_head]
scores = Q @ K.T / sqrt(d_head) -> [T, T]
```

Interpretation: `scores[i, j]` measures how aligned token `i`'s query is with token `j`'s key.

## By-Hand Work First

Do not skip this. The by-hand computation gives you a correctness check for the PyTorch code.

Use:

```text
T = 3 tokens
d_model = 2
d_head = 2
```

Choose tiny `X`, `W_Q`, and `W_K`. Use small integers so the matrix multiplications are easy.

Compute:

1. `Q = X W_Q`
2. `K = X W_K`
3. `Q K^T`
4. `Q K^T / sqrt(d_head)`
5. Which token attends most to which token, based on the largest score in each row?

### Your chosen tensors

Write your chosen `X`, `W_Q`, and `W_K` here before coding.

### Your by-hand result

Write your hand-computed `Q`, `K`, raw `QK^T`, scaled scores, and row-wise highest-attended token.

## Setup

Import PyTorch and choose whether to use `math.sqrt` or `torch.sqrt`.

API you may need:

```python
import math
import torch
```

In [2]:
import math
import torch

## Exercise 1: Unbatched Single Sequence

Implement the score computation for one sequence, matching the math you did by hand.

Use your exact hand-chosen values for `X`, `W_Q`, and `W_K` so you can compare the output to your hand-computed scores.

Expected shapes:

```text
X:        [T, d_model]
W_Q:      [d_model, d_head]
W_K:      [d_model, d_head]
Q:        [T, d_head]
K:        [T, d_head]
K.T:      [d_head, T]
scores:   [T, T]
```

API you may need:

```python
torch.tensor([...], dtype=torch.float32)
X @ W_Q
X @ W_K
K.T
Q @ K.T
math.sqrt(d_head)
scores.shape
torch.argmax(scores, dim=1)
```

In [ ]:
# TODO: define T, d_model, d_head
# TODO: create X, W_Q, W_K using your by-hand values
# TODO: compute Q
# TODO: compute K
# TODO: compute raw_scores = Q @ K.T
# TODO: compute scores = raw_scores / sqrt(d_head)
# TODO: print shapes and compare to your by-hand result

In [11]:
T = 3 # input is 3 tokens
d_model = 2 # model is 2d
d_head = 2 # head is 2d

In [5]:
X = torch.tensor([[1,0], [1,1], [0,2]], dtype=torch.float32)
W_Q = torch.tensor([[1,1], [0,1]], dtype=torch.float32)
W_K = torch.tensor([[1,0], [1,1]], dtype=torch.float32)

In [6]:
# query - what question this token is asking?
Q = X @ W_Q 
print(Q)

tensor([[1., 1.],
        [1., 2.],
        [0., 2.]])


In [8]:
# key - what answer this token can provide?
K = X @ W_K
print(K)

tensor([[1., 0.],
        [2., 1.],
        [2., 2.]])


In [9]:
raw_scores = Q @ K.T # transpost of K so we get a 3x3 matrix here Tokens x Tokens
print(raw_scores)

tensor([[1., 3., 4.],
        [1., 4., 6.],
        [0., 2., 4.]])


In [12]:
scores = raw_scores / math.sqrt(d_head)
print(scores)

tensor([[0.7071, 2.1213, 2.8284],
        [0.7071, 2.8284, 4.2426],
        [0.0000, 1.4142, 2.8284]])


In [14]:
# index of token
torch.argmax(scores, dim=1)

tensor([2, 2, 2])

## Shape Table

Fill this in after Exercise 1.

```text
X:
W_Q:
W_K:
Q:
K:
raw_scores:
scores:
```

In [17]:
print(X.shape)
print(W_Q.shape)
# etc
# understand the shapes
# X is our input, each row is our input embedding  vector
# Weights are our model size
# Q, K are the values per input in our model size dimensinos
# scores are Token x Token

torch.Size([3, 2])
torch.Size([2, 2])


## Understanding the Score Grid

The scores matrix has shape `[T, T]`.

Read it as:

```text
rows = query positions
columns = key positions
scores[i, j] = how much token i's query aligns with token j's key
```

For each row, identify the largest score. That tells you which key position that query position scores highest against.

In [18]:
# TODO: use torch.argmax(scores, dim=1)
# TODO: explain which token each row attends to most
print(scores)
torch.argmax(scores, dim=1)
# token at index 2 (the last one)

tensor([[0.7071, 2.1213, 2.8284],
        [0.7071, 2.8284, 4.2426],
        [0.0000, 1.4142, 2.8284]])


tensor([2, 2, 2])

## Optional: Softmax Preview

Softmax turns raw scores into routing weights. Each row becomes a probability distribution over key positions.

This is a preview only. The main goal of this unit is the score computation.

API you may need:

```python
weights = torch.softmax(scores, dim=-1)
weights.sum(dim=-1)
```

In [22]:
# TODO: apply softmax to scores
# TODO: verify each row sums to 1
weights = torch.softmax(scores, dim=-1)
print(weights)
print(weights.sum(dim=-1))

tensor([[0.0743, 0.3057, 0.6200],
        [0.0229, 0.1911, 0.7860],
        [0.0454, 0.1867, 0.7679]])
tensor([1., 1., 1.])
